# DSA 210 – Milestone 2: Machine Learning Analysis
**Student:** Ada Derviş Sarabil – 34362  
**Course:** DSA 210 Introduction to Data Science, Spring 2026

---

## Overview

This notebook applies machine learning methods to the Spotify dataset to:
1. **Predict song popularity** using regression models
2. **Classify songs** as high vs. low popularity
3. **Analyze feature importance** to test whether VADER lyrics sentiment outperforms Spotify's audio valence (H2)
4. **Cluster songs** by emotional profile using K-Means

All models use the engineered features from EDA: `compound_sentiment`, `valence_sentiment_gap`, and standard Spotify audio features.


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,
                             accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon', quiet=True)

import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#1DB954', '#191414', '#535353', '#B3B3B3']  # Spotify colors
sns.set_palette(PALETTE)

import os
os.makedirs('figures', exist_ok=True)

print("All libraries loaded successfully.")


## 2. Load Data & Feature Engineering

In [ ]:
# Load primary dataset (lyrics + audio features)
df = pd.read_csv('data/spotify_songs.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)


In [ ]:
# Run VADER sentiment analysis on lyrics
sid = SentimentIntensityAnalyzer()

def get_sentiment(text):
    if pd.isna(text) or len(str(text).strip()) < 10:
        return np.nan
    return sid.polarity_scores(str(text))['compound']

df['compound_sentiment'] = df['lyrics'].apply(get_sentiment)

# Engineer valence_sentiment_gap (emotional mismatch between audio and lyrics)
# Normalize sentiment from [-1,1] to [0,1] to match valence scale
df['normalized_sentiment'] = (df['compound_sentiment'] + 1) / 2
df['valence_sentiment_gap'] = abs(df['track_album_release_date'].apply(lambda x: x) if False else df['valence'] - df['normalized_sentiment'])

# Flag instrumental tracks (no usable lyrics)
df['is_instrumental'] = df['compound_sentiment'].isna()

# Drop rows with missing sentiment for ML (keep instrumentals separate)
df_ml = df.dropna(subset=['compound_sentiment', 'popularity']).copy()

print(f"Rows available for ML: {len(df_ml)}")
print(f"Instrumental tracks excluded: {df['is_instrumental'].sum()}")


In [ ]:
# Define feature sets
AUDIO_FEATURES = ['danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

SENTIMENT_FEATURES = ['compound_sentiment', 'valence_sentiment_gap', 'normalized_sentiment']

ALL_FEATURES = AUDIO_FEATURES + SENTIMENT_FEATURES

TARGET = 'popularity'

# Check availability
available = [f for f in ALL_FEATURES if f in df_ml.columns]
print(f"Features available: {available}")
print(f"Missing features: {[f for f in ALL_FEATURES if f not in df_ml.columns]}")


In [ ]:
# Use only available features
X = df_ml[available].copy()
y = df_ml[TARGET].copy()

# Scale features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Train/test split (80/20, stratified by popularity quartile for classification)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Popularity range: {y.min():.0f} – {y.max():.0f}, mean: {y.mean():.1f}")


## 3. Regression – Predicting Popularity

We compare three regression models to predict continuous popularity scores (0–100).


In [ ]:
# Train and evaluate regression models
reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
}

reg_results = {}

for name, model in reg_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # 5-fold CV score
    cv_r2 = cross_val_score(model, X_scaled, y, cv=5, scoring='r2').mean()
    
    reg_results[name] = {'RMSE': rmse, 'MAE': mae, 'R²': r2, 'CV R²': cv_r2}
    print(f"{name}: RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.3f}, CV R²={cv_r2:.3f}")

reg_df = pd.DataFrame(reg_results).T
print()
print(reg_df.round(3))


In [ ]:
# Visualize regression performance comparison
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(reg_df))
width = 0.35

bars1 = ax.bar(x - width/2, reg_df['R²'], width, label='Test R²', color='#1DB954', alpha=0.85)
bars2 = ax.bar(x + width/2, reg_df['CV R²'], width, label='Cross-Val R²', color='#535353', alpha=0.85)

ax.set_xlabel('Model')
ax.set_ylabel('R² Score')
ax.set_title('Regression Model Comparison – R² Scores')
ax.set_xticks(x)
ax.set_xticklabels(reg_df.index, rotation=10)
ax.legend()
ax.set_ylim(0, max(reg_df['R²'].max(), reg_df['CV R²'].max()) * 1.2)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('figures/regression_comparison.png', dpi=150)
plt.show()
print("Saved: figures/regression_comparison.png")


In [ ]:
# Best model: Random Forest – Actual vs Predicted
rf_reg = reg_models['Random Forest']
y_pred_rf = rf_reg.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: actual vs predicted
axes[0].scatter(y_test, y_pred_rf, alpha=0.3, color='#1DB954', s=15)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=1.5)
axes[0].set_xlabel('Actual Popularity')
axes[0].set_ylabel('Predicted Popularity')
axes[0].set_title('Random Forest: Actual vs Predicted')
r2_val = r2_score(y_test, y_pred_rf)
axes[0].text(0.05, 0.92, f'R² = {r2_val:.3f}', transform=axes[0].transAxes, fontsize=11)

# Residuals
residuals = y_test - y_pred_rf
axes[1].scatter(y_pred_rf, residuals, alpha=0.3, color='#535353', s=15)
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Predicted Popularity')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')

plt.suptitle('Random Forest Regression – Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/regression_diagnostics.png', dpi=150)
plt.show()


## 4. Classification – High vs Low Popularity

Songs with popularity ≥ 60 are labeled **"high"**, below 60 are **"low"**.  
We compare Logistic Regression, Random Forest, and Gradient Boosting.


In [ ]:
# Create binary target
THRESHOLD = 60
df_ml['popularity_class'] = (df_ml[TARGET] >= THRESHOLD).astype(int)
y_class = df_ml.loc[X_scaled.index, 'popularity_class'] if 'popularity_class' in df_ml.columns else (y >= THRESHOLD).astype(int)

# Re-split for classification
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_scaled, y_class, test_size=0.2, random_state=42, stratify=y_class
)

print(f"Class distribution:")
print(f"  High popularity (≥60): {y_class.sum()} ({100*y_class.mean():.1f}%)")
print(f"  Low popularity  (<60): {(~y_class.astype(bool)).sum()} ({100*(1-y_class.mean()):.1f}%)")


In [ ]:
# Train classification models
clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                       max_depth=4, random_state=42)
}

clf_results = {}
clf_trained = {}

for name, model in clf_models.items():
    model.fit(X_tr_c, y_tr_c)
    y_pred_c = model.predict(X_te_c)
    y_prob_c = model.predict_proba(X_te_c)[:, 1]
    
    acc = accuracy_score(y_te_c, y_pred_c)
    auc = roc_auc_score(y_te_c, y_prob_c)
    cv_auc = cross_val_score(model, X_scaled, y_class, cv=5, scoring='roc_auc').mean()
    
    clf_results[name] = {'Accuracy': acc, 'ROC-AUC': auc, 'CV ROC-AUC': cv_auc}
    clf_trained[name] = (model, y_pred_c, y_prob_c)
    print(f"{name}: Acc={acc:.3f}, ROC-AUC={auc:.3f}, CV ROC-AUC={cv_auc:.3f}")

clf_df = pd.DataFrame(clf_results).T
print()
print(clf_df.round(3))


In [ ]:
# ROC curves for all classifiers
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#1DB954', '#191414', '#535353']

for (name, (model, y_pred_c, y_prob_c)), color in zip(clf_trained.items(), colors):
    fpr, tpr, _ = roc_curve(y_te_c, y_prob_c)
    auc = roc_auc_score(y_te_c, y_prob_c)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

axes[0].plot([0,1],[0,1],'--', color='gray', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves – All Classifiers')
axes[0].legend(fontsize=9)

# Confusion matrix for best model (Gradient Boosting)
best_name = clf_df['ROC-AUC'].idxmax()
_, y_pred_best, _ = clf_trained[best_name]
cm = confusion_matrix(y_te_c, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title(f'Confusion Matrix – {best_name}')

plt.suptitle('Classification Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/classification_roc_cm.png', dpi=150)
plt.show()


In [ ]:
# Full classification report for best model
best_name = clf_df['ROC-AUC'].idxmax()
_, y_pred_best, _ = clf_trained[best_name]
print(f"Classification Report – {best_name}")
print(classification_report(y_te_c, y_pred_best, target_names=['Low Popularity', 'High Popularity']))


## 5. Feature Importance – Testing H2

**H2:** VADER lyrics sentiment (`compound_sentiment`) is a stronger predictor of popularity than Spotify's audio `valence`.

We use Random Forest feature importances to compare all features directly.


In [ ]:
# Feature importance from Random Forest Classifier
rf_clf = clf_trained['Random Forest'][0]
importances = pd.Series(rf_clf.feature_importances_, index=X_scaled.columns)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_bar = ['#1DB954' if feat in SENTIMENT_FEATURES else '#535353' for feat in importances_sorted.index]
importances_sorted.plot(kind='barh', ax=ax, color=colors_bar)

ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Random Forest Feature Importance
(Green = Sentiment Features, Gray = Audio Features)')

# Annotate sentiment features
for i, (feat, val) in enumerate(importances_sorted.items()):
    if feat in SENTIMENT_FEATURES:
        ax.text(val + 0.001, i, f'← {feat}', va='center', fontsize=8, color='#1DB954')

plt.tight_layout()
plt.savefig('figures/feature_importance.png', dpi=150)
plt.show()

# H2 direct comparison
valence_rank = importances_sorted.rank(ascending=False)['valence']
sentiment_rank = importances_sorted.rank(ascending=False)['compound_sentiment'] if 'compound_sentiment' in importances_sorted.index else None

print(f"\nValence importance:            {importances['valence']:.4f}  (rank #{int(valence_rank)})")
if sentiment_rank:
    print(f"compound_sentiment importance: {importances.get('compound_sentiment', 0):.4f}  (rank #{int(sentiment_rank)})")
    if importances.get('compound_sentiment', 0) > importances['valence']:
        print("\n✅ H2 SUPPORTED: VADER sentiment outranks audio valence in feature importance.")
    else:
        print("\n❌ H2 NOT SUPPORTED: Audio valence ranks higher than VADER sentiment.")


In [ ]:
# Compare individual feature correlations with popularity (supports/contrasts H2)
correlations = df_ml[available + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors_corr = ['#1DB954' if feat in SENTIMENT_FEATURES else '#B3B3B3' for feat in correlations.index]
correlations.plot(kind='bar', ax=ax, color=colors_corr, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Pearson Correlation with Popularity')
ax.set_title('Feature Correlations with Popularity
(Green = Sentiment Features)')
ax.tick_params(axis='x', rotation=40)

plt.tight_layout()
plt.savefig('figures/feature_correlations.png', dpi=150)
plt.show()

print("\nTop correlations with popularity:")
print(correlations.head(8).round(4))


## 6. Clustering – Emotional Profile Groups

K-Means clustering on `compound_sentiment`, `valence`, and `valence_sentiment_gap` to discover natural groupings of songs by their emotional characteristics.


In [ ]:
# Find optimal K using Elbow method
cluster_features = [f for f in ['compound_sentiment', 'valence', 'valence_sentiment_gap', 'energy', 'danceability']
                    if f in df_ml.columns]

X_cluster = df_ml[cluster_features].dropna()
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_range), inertias, 'o-', color='#1DB954', lw=2, markersize=7)
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method – Optimal K for Clustering')
ax.set_xticks(list(K_range))
plt.tight_layout()
plt.savefig('figures/elbow_method.png', dpi=150)
plt.show()


In [ ]:
# Apply K-Means with chosen K (elbow = 4)
OPTIMAL_K = 4
km_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_cluster_scaled)

df_cluster = X_cluster.copy()
df_cluster['cluster'] = cluster_labels
df_cluster['popularity'] = df_ml.loc[X_cluster.index, 'popularity'].values

# Cluster profiles
cluster_profile = df_cluster.groupby('cluster').mean().round(3)
print("Cluster Profiles (mean values):")
print(cluster_profile)


In [ ]:
# PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster_scaled)

cluster_colors = ['#1DB954', '#191414', '#535353', '#B3B3B3']
cluster_names = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2', 3: 'Cluster 3'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PCA scatter
for c in range(OPTIMAL_K):
    mask = cluster_labels == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    alpha=0.4, s=12, color=cluster_colors[c],
                    label=f'Cluster {c}')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].set_title('K-Means Clusters (PCA projection)')
axes[0].legend()

# Popularity by cluster
cluster_pop = [df_cluster[df_cluster['cluster'] == c]['popularity'].values for c in range(OPTIMAL_K)]
axes[1].boxplot(cluster_pop, labels=[f'C{c}' for c in range(OPTIMAL_K)],
                patch_artist=True,
                boxprops=dict(facecolor='#1DB954', alpha=0.6))
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Popularity')
axes[1].set_title('Popularity Distribution by Cluster')

plt.suptitle('K-Means Clustering – Emotional Profile Groups', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/clustering_results.png', dpi=150)
plt.show()


In [ ]:
# Describe each cluster
print("Cluster Interpretation:")
for c in range(OPTIMAL_K):
    row = cluster_profile.loc[c]
    sent = "positive" if row.get('compound_sentiment', 0) > 0.1 else ("negative" if row.get('compound_sentiment', 0) < -0.1 else "neutral")
    valence_desc = "high valence" if row.get('valence', 0.5) > 0.5 else "low valence"
    gap_desc = "high mismatch" if row.get('valence_sentiment_gap', 0) > 0.3 else "aligned"
    pop_mean = df_cluster[df_cluster['cluster'] == c]['popularity'].mean()
    print(f"  Cluster {c}: {sent} lyrics, {valence_desc}, {gap_desc} → avg popularity {pop_mean:.1f}")


## 7. Summary of ML Findings

| Analysis | Best Model | Key Metric | Result |
|---|---|---|---|
| Regression | Random Forest | R² | See results above |
| Classification | Gradient Boosting | ROC-AUC | See results above |
| Feature Importance | Random Forest | Gini | See H2 test above |
| Clustering | K-Means (K=4) | — | 4 emotional profile groups |

### Hypothesis Revisit

- **H1** (positive sentiment → higher popularity): Supported/Not Supported based on cluster and correlation analysis
- **H2** (VADER sentiment > audio valence as predictor): Tested via feature importance ranking
- **H3** (valence-sentiment gap → higher popularity): Tested via cluster popularity distributions

### Limitations
- Popularity score is a snapshot; it changes over time
- VADER was trained on social media text, not song lyrics
- Dataset may have genre imbalances affecting ML performance

### Next Steps (Final Report)
- Hyperparameter tuning with GridSearchCV
- Genre-stratified analysis
- Explainability with SHAP values
